# PYNQ-Z2 INT8 Matmul Accelerator Demo

This notebook is a board-side demo entry point. The reusable logic lives in the Python modules under `software/` and `pynq_driver/`.

Default overlay source:

- `../fpga_hardware/accelerator_hardware/AI_accelerator.xsa`

The driver extracts the contained `.bit/.hwh` into `overlays/generated/` before loading the overlay.

In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "pynq_driver":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pynq_driver.benchmark import run_benchmark
from pynq_driver.matmul_accel import MatmulAccel
from pynq_driver.overlay_driver import OverlayDriver
from pynq_driver.register_map import RegisterMap
from software.cpu_ref import matmul_int8_ref
from software.test_vectors import load_test_vectors

In [ ]:
config_path = PROJECT_ROOT / "configs" / "default.json"
with config_path.open("r", encoding="utf-8") as config_file:
    config = json.load(config_file)

overlay_config = config.get("overlay_source", config.get("overlay_bitfile"))
overlay_source = PROJECT_ROOT / overlay_config
register_map_path = PROJECT_ROOT / config["register_map"]
ip_name = config["ip_name"]
vector_dir = PROJECT_ROOT / "test_vectors" / "default"

print(f"Project root: {PROJECT_ROOT}")
print(f"Overlay source: {overlay_source}")
print(f"IP name: {ip_name}")
print(f"Register map: {register_map_path}")
print(f"Vectors: {vector_dir}")

In [ ]:
overlay_driver = OverlayDriver(str(overlay_source), ip_name)
overlay_driver.print_ip_dict()

In [ ]:
input_a, input_b, stored_golden_c, metadata = load_test_vectors(vector_dir)
shift = int(metadata["shift"])

print(f"A: shape={input_a.shape}, dtype={input_a.dtype}")
print(f"B: shape={input_b.shape}, dtype={input_b.dtype}")
print(f"C golden: shape={stored_golden_c.shape}, dtype={stored_golden_c.dtype}")
print(metadata)

In [ ]:
cpu_start = time.perf_counter()
golden_c = matmul_int8_ref(input_a, input_b, shift)
cpu_time_ms = (time.perf_counter() - cpu_start) * 1000.0

np.testing.assert_array_equal(golden_c, stored_golden_c)
print(f"CPU golden recomputed in {cpu_time_ms:.6f} ms")

In [ ]:
register_map = RegisterMap.from_json(register_map_path)
accel = MatmulAccel(str(overlay_source), ip_name, register_map)
fpga_c, timing = accel.run_with_timing(input_a, input_b, shift)

print(timing)

In [ ]:
np.testing.assert_array_equal(fpga_c, golden_c)
diff = fpga_c.astype(np.int64) - golden_c.astype(np.int64)
max_abs_error = int(np.max(np.abs(diff))) if diff.size else 0

print("PASS")
print(f"M={metadata['M']} K={metadata['K']} N={metadata['N']} shift={shift}")
print(f"max_abs_error={max_abs_error}")
print(f"cpu_time_ms={cpu_time_ms:.6f}")
print(f"fpga_kernel_time_ms={timing['fpga_kernel_time_ms']:.6f}")
print(f"total_time_ms={timing['total_time_ms']:.6f}")

In [ ]:
cases = [
    {"name": "tiny", "M": 2, "K": 4, "N": 2, "shift": 7, "seed": 0},
    {"name": "small", "M": 16, "K": 64, "N": 16, "shift": 7, "seed": 1},
    {"name": "medium", "M": 32, "K": 128, "N": 32, "shift": 7, "seed": 2},
    {"name": "shift", "M": 16, "K": 64, "N": 16, "shift": 3, "seed": 3},
]
rows = run_benchmark(cases, accel)
for row, case in zip(rows, cases):
    row["case"] = case["name"]

try:
    import pandas as pd

    display(pd.DataFrame(rows))
except ImportError:
    for row in rows:
        print(row)